# Homework 02: The Muscle Speed Limit and Directional Accuracy

**Computational Sensorimotor Control** | Week 2 | Module 1: The Biological Plant

**Due:** One week from assignment | **Total:** 100 points

---

## The Experiment

A participant makes fast 10 cm reaches from a central start posture (θ₁ = 55°, θ₂ = 75°) to each of 8 equally spaced targets, all at the same hand speed. The experimenter measures endpoint accuracy in each direction.

In HW01, you found that different reach directions require different joint excursions because of the Jacobian. In this homework, you will discover that the **calcium kinetics delay** (from Lab 02) converts these direction-dependent joint velocities into **direction-dependent endpoint accuracy**.

**The logic chain:**
1. Same hand speed → different joint speeds (because of the Jacobian)
2. Same calcium delay → different joint-space overshoots (faster joints overshoot more)
3. Nonlinear FK maps joint overshoots back to task space → direction-dependent hand accuracy

You will need your `Arm` class from Lab 01 and your `Muscle` class from Lab 02.

---
## Part 0: Setup (0 pts)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

c = 0.112; G = 20.0; TAU = 0.015  # G = neural drive gain (mm)
f1, f2, f3, f4 = 0.82, 0.50, 0.43, 58.0

# Joint limits (from Week 1)
T1_MIN, T1_MAX = 0, np.radians(140)   # shoulder: 0°–140°
T2_MIN, T2_MAX = 0, np.radians(145)   # elbow: 0°–145°

print('Setup complete.')

### Paste Your Classes
Paste your `Arm` class (Lab 01) and your `Muscle` class with supporting functions (Lab 02) below.

In [ ]:
# PASTE your Arm class and Muscle class here



---
## Part 1: Characterize the Muscle Speed Limit (20 pts)

Before predicting directional accuracy, we need to measure how fast muscles can start and stop producing force.

### Task 1.1 — Step Response: How Fast Can Force Rise? (5 pts)

COMPLETE the cell below to simulate the pectoralis (ρ = 14.9, k = 258.5, moment arm = +0.04 m, rest length = 0.26 m) responding to a step activation from a = 0 to a = 0.8 at t = 50 ms. The muscle is isometric (at resting length, zero velocity). Measure the time to 50%, 90%, and 99% of steady-state force.

In [ ]:
dt = 0.0001; T = 0.3; t = np.arange(0, T, dt)

# TODO: COMPLETE — create the pectoralis muscle and reset it
pec = ...  # your code here

activation = np.where(t >= 0.05, 0.8, 0.0)

# TODO: COMPLETE — simulate, record force at each timestep
forces_step = np.zeros_like(t)
for i in range(len(t)):
    forces_step[i] = ...  # your code here

# Measure rise times
ss_force = np.mean(forces_step[-1000:])
print(f"Steady-state force: {ss_force:.4f} N")
for pct in [0.50, 0.90, 0.99]:
    threshold = pct * ss_force
    idx = np.argmax(forces_step >= threshold)
    rise_time = (t[idx] - 0.05) * 1000
    print(f"  Time to {pct*100:.0f}%: {rise_time:.0f} ms after command")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t*1000, activation, 'k', lw=2); ax1.set_ylabel('Activation')
ax1.set_title('Step Response: Command vs. Force')
ax2.plot(t*1000, forces_step, '#E74C3C', lw=2.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Force (N)')
plt.tight_layout(); plt.show()

### Task 1.2 — Braking Response: How Fast Can Force Disappear? (5 pts)

Now the critical question for movement control: how fast does force drop when the brain says "stop"?

COMPLETE the cell below to simulate a 100 ms activation pulse (a = 0.8 from 50–150 ms), then braking (a = 0). Measure the time from braking command to 10% of peak force.

In [ ]:
dt = 0.0001; T = 0.4; t = np.arange(0, T, dt)

pec.reset()  # reuse the same muscle object

activation = np.zeros_like(t)
activation[(t >= 0.05) & (t < 0.15)] = 0.8

# TODO: COMPLETE — simulate and record force
forces_brake = np.zeros_like(t)
for i in range(len(t)):
    forces_brake[i] = ...  # your code here

# Measure peak and decay
peak_force = np.max(forces_brake)
peak_time_ms = t[np.argmax(forces_brake)] * 1000

brake_idx = int(0.15 / dt)
post_brake = forces_brake[brake_idx:]
idx_10 = np.argmax(post_brake <= 0.1 * peak_force)
decay_time_ms = idx_10 * dt * 1000

print(f"Peak force: {peak_force:.4f} N at t = {peak_time_ms:.0f} ms")
print(f"Time to 10% of peak: {decay_time_ms:.0f} ms after braking command")

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t*1000, activation, 'k', lw=2); ax1.set_ylabel('Activation')
ax1.axvline(150, color='red', ls='--', alpha=0.5, label='Braking command'); ax1.legend()
ax1.set_title('Braking Response')
ax2.plot(t*1000, forces_brake, '#E74C3C', lw=2.5)
ax2.axvline(150, color='red', ls='--', alpha=0.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Force (N)')
plt.tight_layout(); plt.show()

### Task 1.3 — What Does the Delay Cost? (5 pts)

COMPLETE the cell below to compute how far the hand travels during the braking delay at a typical reaching velocity of 0.5 m/s. This is the **overshoot** — the distance the hand moves past its intended stopping point because the braking force develops too slowly.

In [ ]:
# TODO: COMPLETE — compute overshoot
calcium_delay = decay_time_ms / 1000  # convert your measured delay to seconds
v_hand = 0.5  # m/s (typical peak velocity for a 10 cm reach in 200 ms)

overshoot_m = ...   # your code here: v_hand * calcium_delay
overshoot_cm = ...  # your code here: convert to cm

print(f"Calcium delay: {calcium_delay*1000:.0f} ms")
print(f"Hand velocity: {v_hand} m/s")
print(f"Predicted overshoot: {overshoot_cm:.1f} cm")
print(f"For a 10 cm reach, that is {overshoot_cm/10*100:.0f}% of the target distance")

### Question 1.1 (5 pts)

**(a)** Why does force continue to rise briefly *after* the braking command at 150 ms? (Hint: what is happening inside the calcium filter when the input drops to zero?)

**(b)** At 0.5 m/s, the overshoot is ~3 cm for a 10 cm reach. Humans achieve ~1 cm accuracy. What must the brain do instead of waiting until the hand reaches the target to send the braking command?

*Your answer here:*


---
## Part 2: Set Up the Reaching Experiment (15 pts)

Now connect the calcium delay to the center-out reaching experiment from HW01.

### Task 2.1 — Place the 8 Targets and Verify Reachability (5 pts)

COMPLETE the cell below to set up the start posture, compute the 8 target positions, and verify all are reachable (reuse your IK code from HW01).

In [ ]:
arm = Arm()
q_start = np.array([np.radians(55), np.radians(75)])
start_pos = arm.forward_kinematics(q_start)

reach_dist = 0.10  # 10 cm
directions = np.arange(0, 360, 45)

# TODO: COMPLETE — compute target positions and check reachability
targets = {}
for d in directions:
    target_pos = ...  # your code here: start_pos + reach_dist * [cos, sin]
    sols = arm.inverse_kinematics(target_pos)
    reachable = len(sols) > 0
    targets[d] = {'pos': target_pos, 'reachable': reachable, 'q_target': sols[0] if reachable else None}
    print(f"{d:3d}°: pos=({target_pos[0]:.4f}, {target_pos[1]:.4f}), reachable={reachable}")

n_reachable = sum(1 for v in targets.values() if v['reachable'])
print(f"\n{n_reachable}/8 targets reachable")

### Task 2.2 — Walk Through ONE Direction: 0° (10 pts)

Before looping over all 8 directions, work through one direction step by step. This builds your understanding of the logic chain.

For a reach toward the 0° target (rightward):
1. Get the target joint configuration from IK
2. Compute the joint displacement Δq = q_target − q_start
3. Compute the total joint excursion ‖Δq‖
4. Compute the peak joint velocity, assuming a bell-shaped (minimum-jerk) velocity profile where peak ≈ 2 × mean: **dq_peak = 2 × Δq / movement_time**
5. Print everything

COMPLETE the cell below.

In [ ]:
movement_time = 0.20  # 200 ms for all reaches

# Work through the 0° direction step by step
d = 0
q_target = targets[d]['q_target']

# Step 1: Joint displacement
# TODO: COMPLETE
delta_q = ...  # your code here: q_target - q_start

# Step 2: Total joint excursion
# TODO: COMPLETE
excursion_rad = ...  # your code here: np.linalg.norm(delta_q)

# Step 3: Peak joint velocity (bell-shaped profile: peak ≈ 2 × mean)
# TODO: COMPLETE
dq_peak = ...  # your code here: 2 * delta_q / movement_time  (vector, rad/s)

# Print
print(f"=== Direction: {d}° ===")
print(f"Start:  θ₁ = {np.degrees(q_start[0]):.1f}°, θ₂ = {np.degrees(q_start[1]):.1f}°")
print(f"Target: θ₁ = {np.degrees(q_target[0]):.1f}°, θ₂ = {np.degrees(q_target[1]):.1f}°")
print(f"Δq:     Δθ₁ = {np.degrees(delta_q[0]):.1f}°, Δθ₂ = {np.degrees(delta_q[1]):.1f}°")
print(f"‖Δq‖ = {np.degrees(excursion_rad):.1f}°")
print(f"Peak joint velocity: dθ₁/dt = {np.degrees(dq_peak[0]):.1f} °/s, dθ₂/dt = {np.degrees(dq_peak[1]):.1f} °/s")
print(f"‖dq/dt‖_peak = {np.degrees(np.linalg.norm(dq_peak)):.1f} °/s")

---
## Part 3: Direction-Dependent Joint Demands (15 pts)

### Task 3.1 — Compute Joint Velocities for All 8 Directions (5 pts)

Now extend your single-direction computation to all 8 targets.

COMPLETE the cell below — the structure is the same as Task 2.2, but in a loop.

In [ ]:
results = {}
print(f"{'Dir':>5s} {'‖Δq‖ (°)':>10s} {'Peak ‖dq/dt‖ (°/s)':>20s} {'dθ₁/dt (°/s)':>14s} {'dθ₂/dt (°/s)':>14s}")
print("-" * 68)

for d in directions:
    if not targets[d]['reachable']:
        print(f"{d:>5d}°  UNREACHABLE")
        continue
    
    q_target = targets[d]['q_target']
    
    # TODO: COMPLETE — same computation as Task 2.2
    delta_q = ...
    dq_peak = ...
    excursion_deg = np.degrees(np.linalg.norm(delta_q))
    dq_peak_mag = np.degrees(np.linalg.norm(dq_peak))
    
    results[d] = {
        'q_target': q_target,
        'delta_q': delta_q,
        'dq_peak': dq_peak,
        'dq_peak_mag': dq_peak_mag,
    }
    
    print(f"{d:>5d}°  {excursion_deg:>9.1f}  {dq_peak_mag:>18.1f}  {np.degrees(dq_peak[0]):>13.1f}  {np.degrees(dq_peak[1]):>13.1f}")

### Task 3.2 — Polar Plot of Joint Speed (5 pts)

COMPLETE the cell below to create a polar plot of peak joint speed vs. direction.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

angles = np.radians(list(results.keys()))
speeds = [results[d]['dq_peak_mag'] for d in results]

# Close the polygon
angles_closed = np.append(angles, angles[0])
speeds_closed = np.append(speeds, speeds[0])

# TODO: COMPLETE — plot
ax.plot(angles_closed, speeds_closed, '#2E86AB', lw=2.5, marker='o', ms=8)
ax.fill(angles_closed, speeds_closed, alpha=0.15, color='#2E86AB')

ax.set_title('Peak Joint Speed by Reach Direction\n(larger = harder for joints)', pad=20, fontweight='bold')
plt.show()

### Question 3.1 (5 pts)

Which direction requires the fastest joint velocities? Is it the same direction that HW01 identified as "hard" (largest joint excursion)? Why or why not?

*Your answer here:*


---
## Part 4: Predict Direction-Dependent Overshoot (25 pts)

Now the key step: the joints overshoot because the calcium delay prevents instantaneous braking. Map this overshoot through the nonlinear FK to get task-space accuracy.

### Task 4.1 — Walk Through ONE Direction: 0° (5 pts)

Before looping, work through the 0° direction step by step:
1. Compute the overshot joint configuration: **q_overshot = q_target + dq_peak × calcium_delay**
2. Check that q_overshot is within joint limits [0°, 140°] × [0°, 145°]
3. Map back to task space: hand_overshot = arm.forward_kinematics(q_overshot)
4. Compute the hand overshoot distance

COMPLETE the cell below.

In [ ]:
d = 0  # work through 0° direction
data = results[d]

# Step 1: Overshot joint configuration
# TODO: COMPLETE
q_overshot = ...  # your code here: data['q_target'] + data['dq_peak'] * calcium_delay

# Step 2: Check joint limits
print(f"=== Direction: {d}° ===")
print(f"Target joint config:   θ₁ = {np.degrees(data['q_target'][0]):.1f}°, θ₂ = {np.degrees(data['q_target'][1]):.1f}°")
print(f"Overshot joint config: θ₁ = {np.degrees(q_overshot[0]):.1f}°, θ₂ = {np.degrees(q_overshot[1]):.1f}°")

valid = (T1_MIN <= q_overshot[0] <= T1_MAX) and (T2_MIN <= q_overshot[1] <= T2_MAX)
print(f"Within joint limits [0°,140°]×[0°,145°]: {valid}")

# Step 3: Map to task space
hand_target = arm.forward_kinematics(data['q_target'])
hand_overshot = arm.forward_kinematics(q_overshot)

# Step 4: Overshoot distance
# TODO: COMPLETE
overshoot_cm = ...  # your code here: np.linalg.norm(hand_overshot - hand_target) * 100

print(f"Hand at target:   ({hand_target[0]:.4f}, {hand_target[1]:.4f}) m")
print(f"Hand overshot to: ({hand_overshot[0]:.4f}, {hand_overshot[1]:.4f}) m")
print(f"Task-space overshoot: {overshoot_cm:.2f} cm")

### Task 4.2 — All 8 Directions with Joint-Limit Check (15 pts)

COMPLETE the cell below to compute the overshoot for all 8 directions. For each direction, check whether the overshot configuration is within joint limits. Flag any violations.

In [ ]:
print(f"{'Dir':>5s} {'Joint overshoot (°)':>20s} {'Hand overshoot (cm)':>20s} {'Limits OK':>10s}")
print("-" * 60)

for d, data in results.items():
    # TODO: COMPLETE — same computation as Task 4.1, in a loop
    q_overshot = ...  # your code here
    
    # Check joint limits
    valid = (T1_MIN <= q_overshot[0] <= T1_MAX) and (T2_MIN <= q_overshot[1] <= T2_MAX)
    
    hand_target = arm.forward_kinematics(data['q_target'])
    hand_overshot = arm.forward_kinematics(q_overshot)
    
    overshoot_cm = ...  # your code here
    joint_overshoot_deg = np.degrees(np.linalg.norm(q_overshot - data['q_target']))
    
    data['overshoot_cm'] = overshoot_cm
    data['q_overshot'] = q_overshot
    data['limits_ok'] = valid
    
    flag = "✓" if valid else "✗ VIOLATION"
    print(f"{d:>5d}°  {joint_overshoot_deg:>18.1f}°  {overshoot_cm:>18.2f} cm  {flag:>10s}")

# Summary
os_vals = [data['overshoot_cm'] for data in results.values()]
print(f"\nRange: {min(os_vals):.2f} – {max(os_vals):.2f} cm")
print(f"Ratio (max/min): {max(os_vals)/min(os_vals):.2f}×")

n_valid = sum(1 for data in results.values() if data['limits_ok'])
print(f"Within joint limits: {n_valid}/{len(results)}")

### Task 4.3 — Polar Overshoot Plot (5 pts)

COMPLETE the cell below to create the polar plot of predicted hand overshoot across all 8 directions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))

angles = np.radians(list(results.keys()))
overshoots = [results[d]['overshoot_cm'] for d in results]
angles_closed = np.append(angles, angles[0])
overshoots_closed = np.append(overshoots, overshoots[0])

# TODO: COMPLETE — plot the polar diagram
ax.plot(angles_closed, overshoots_closed, '#E74C3C', lw=2.5, marker='o', ms=8)
ax.fill(angles_closed, overshoots_closed, alpha=0.15, color='#E74C3C')

ax.set_title('Predicted Hand Overshoot by Direction (cm)\n(larger = less accurate)', pad=20, fontweight='bold')
plt.show()

# Report
best_dir = min(results, key=lambda d: results[d]['overshoot_cm'])
worst_dir = max(results, key=lambda d: results[d]['overshoot_cm'])
print(f"Most accurate direction:  {best_dir}° ({results[best_dir]['overshoot_cm']:.2f} cm)")
print(f"Least accurate direction: {worst_dir}° ({results[worst_dir]['overshoot_cm']:.2f} cm)")

### Question 4.1 (included in write-up points)

**(a)** Which direction has the smallest predicted overshoot (most accurate)? Which has the largest? Report the ratio.

**(b)** Is the overshoot pattern simply the inverse of the manipulability ellipsoid from HW01? If not, what additional factor shapes it?

**(c)** Explain the complete logic chain in one or two sentences: why does the Jacobian create direction-dependent accuracy when combined with a fixed muscle delay?

*Your answer here:*


---
## Part 5: Methods & Predictions Write-Up (25 pts)

Write a **Methods and Predictions** section (300–500 words):

**Methods (~150 words):** Describe the arm model, the 8 reach directions, the movement parameters (10 cm, 200 ms), how you measured the calcium delay, how you computed peak joint velocities from the Jacobian, and how you used the nonlinear FK to predict task-space overshoot. Note whether all overshot configurations were within joint limits.

**Predictions (~150–300 words):** Report the direction-dependent overshoot pattern. State which directions are predicted to be most/least accurate and by what ratio. Explain the logic chain (Jacobian → joint velocities → calcium delay → nonlinear FK → directional accuracy). Note that this prediction is consistent with experimental findings of direction-dependent reaching variability (Gordon et al. 1994). Include your polar plot as "Figure 1."

**Format:** Third person, past tense.

*Your write-up here:*


---
## Looking Ahead: Antagonist Braking and the Triphasic Pattern

In this homework, "braking" means shutting off the agonist and waiting for force to decay. But real braking uses the **antagonist** — an active braking burst that opposes the movement.

In **Week 3** (Forward Dynamics), when you first simulate actual reaching, you will discover the **triphasic pattern**: agonist ON → antagonist ON → agonist fine-tune. The calcium delay affects *both* the agonist shutoff and the antagonist rise, so the effective braking window is ~58 ms regardless of strategy.

In **Week 4**, the λ model will automate this entire pattern via the stretch reflex — no explicit timing of agonist/antagonist bursts is needed.

---
## Summary

You combined two weeks of analysis to make a cross-module prediction:

- **Week 1 (Jacobian):** Different reach directions require different joint velocities for the same hand speed
- **Week 2 (Calcium delay):** The ~58 ms calcium filter delay means faster-moving joints overshoot more in joint space
- **Nonlinear FK:** Joint-space overshoots map back to task space through the nonlinear forward kinematics, creating a direction-dependent accuracy pattern

**Key takeaway:** The brain faces direction-dependent control challenges that arise from the body's geometry and the muscles' temporal dynamics. Optimal motor planning (Weeks 12–14) must account for both.